In [177]:
import json

In [178]:
collations_and_orders = {
    "Mc": ["I344", "R2536", "P400", "P3465", "P5881", "P3475", "P3473", "R2407", "BWII672", "A4095", "P3466", "CCCP578", "L8751", "L4044", "P3471"]
}
with open("manuscripts.json", "r", encoding="utf-8") as file:
    manuscripts_src = json.loads(file.read())

with open("units.json", "r", encoding="utf-8") as file:
    units_src = json.loads(file.read())
with open("segments.json", "r", encoding="utf-8") as file:
    segments_src = json.loads(file.read())
with open("pages.json", "r", encoding="utf-8") as file:
    pages_src = json.loads(file.read())
with open("text.json", "r", encoding="utf-8") as file:
    text_src = json.loads(file.read())
with open("lines.json", "r", encoding="utf-8") as file:
    lines_src = json.loads(file.read())
with open("images.json", "r", encoding="utf-8") as file:
    images_src = json.loads(file.read())

In [179]:
def get_mss_siglum_to_id(ms_list):
    res = {}
    for siglum in ms_list:
        for ms in manuscripts_src:
            if siglum == ms['siglum']:
                res[siglum] = ms['id']
    return res
    

In [180]:
def get_units(chapter):
    units = [x for x in units_src if x.get('frame', '') == chapter]
    unit_ids = {x['id'] for x in units}
    return units, unit_ids

In [181]:
def format_units_orders(units, root_unit_title):
    root_unit = next((x for x in units if x['title'] == root_unit_title), None)
    def get_childern(root_id, order):
        children = []
        for unit in units:
            if unit['parentID'] == root_id:
                unit['formattedOrder'] = f"{order}.{unit['order']}"
                children.append(unit)
                children += get_childern(unit['id'], unit['order'])
        return children
    return [root_unit] + get_childern(root_unit['id'], f"{root_unit['order']}")

In [182]:
def get_segments(ms_id, unit_ids):
    segments = [x for x in segments_src if x['unitID'] in unit_ids and x['mediumID'] == ms_id]
    return {x['unitID']: x for x in segments}


In [183]:
def get_segment_pages(ms_id, segment):
    pages = [x for x in pages_src if x['mediumID'] == ms_id and x['number']>= segment['startPage'] and x['number'] <= segment['endPage']  ]
    return pages

def get_segment_bounding_pages(pages, segment):
    first = next((x for x in pages if x['number'] == segment['startPage']), None)
    last = next((x for x in pages if x['number'] == segment['endPage']), None)
    return first, last

In [184]:
def get_page_body_lines(page):
    element_ids = {x['id'] for x in text_src if x['pageID'] == page['id'] and 'main' in x['position']}
    lines = [x for x in lines_src if x['elementID'] in element_ids]
    for line in lines:
        line['pageNumber'] = page['number']
        line['pageID'] = page['id']
    return lines

In [185]:
def get_segment_lines(lines, segment, start_page_id, end_page_id):
    if segment["startPage"] == segment["endPage"]:
        return [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["order"] <= segment["endLine"]
        ]
    if segment["endPage"] - segment["startPage"] == 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageID"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageID"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_end_page
    if segment["endPage"] - segment["startPage"] > 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageID"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_mid_pages = [
            x
            for x in lines
            if x["pageID"] != start_page_id and  x["pageID"] != end_page_id
        ]
        lines_from_mid_pages.sort(key=lambda item: (item["order"], item['pageNumber']))
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageID"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_mid_pages + lines_from_end_page

In [186]:
def get_line_data(line, start=None, end=None):
    if start is not None and end is not None:
        tokens = line['tokens'][start:end]
        states = line['states'][start:end]
    elif start is not None:
        tokens = line['tokens'][start:]
        states = line['states'][start:]
    elif end is not None:
        tokens = line['tokens'][:end]
        states = line['states'][:end]
    else:
        tokens = line['tokens']
        states = line['states']
    
    return {
        'tokens': tokens,
        'states': states,
        'lines': [line['order']] * len(tokens),
        'pages': [line['pageNumber']] * len(tokens),
        'breaks': [None if i != 0 or line['order'] != 0 else line['pageNumber'] 
                   for i in range(len(tokens))]
    }

def get_segment_data_same_line(lines, segment):
    line_data = get_line_data(lines[0], start=segment['startToken'], end=segment['endToken']+1)
    data = {
        'images': [],
        'lemmas': []
    }
    data.update(line_data)
    return data



def get_segment_data_same_page(lines, segment):
    data = {
        'tokens': [],
        'states': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)
        
        data['tokens'] += line_data['tokens']
        data['states'] += line_data['states']
        data['lines'] += line_data['lines']
        data['pages'] += line_data['pages']
        data['breaks'] += line_data['breaks']

    return data


def get_segment_data_multiple_pages(lines, segment):
    data = {
        'tokens': [],
        'states': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine'] and line['pageNumber'] == segment['startPage']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine'] and line['pageNumber'] == segment['endPage']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)
        
        data['tokens'] += line_data['tokens']
        data['states'] += line_data['states']
        data['lines'] += line_data['lines']
        data['pages'] += line_data['pages']
        data['breaks'] += line_data['breaks']

    return data


def get_segment_data(lines, segment):
    if segment['startPage'] == segment['endPage']:
        if segment['startLine'] == segment['endLine']:
            return get_segment_data_same_line(lines, segment)
        else:
            return get_segment_data_same_page(lines, segment)
    else:
        return get_segment_data_multiple_pages(lines, segment)


In [187]:
## todo: REFACTOR
def pipeline(chapter, root_unit):
    mss_siglum_to_id = get_mss_siglum_to_id(collations_and_orders[chapter])
    units, unit_ids = get_units(chapter)
    units = format_units_orders(units, root_unit)
    units.sort(key=lambda item: item["order"])
    data = []
    for i, ms in enumerate(collations_and_orders[chapter]):
        id = mss_siglum_to_id[ms]
        segments = get_segments(mss_siglum_to_id[ms], unit_ids)
        dto = {"siglum": ms, "id": id, "order": i}
        ms_units = []
        facsimiles = []
        for unit in units:
            id = unit["id"]
            if id not in segments or segments[id]["lacuna"]:
                ms_units.append(None)
            else:
                pages = get_segment_pages(mss_siglum_to_id[ms], segments[id])
                first, last = get_segment_bounding_pages(pages, segments[id])
                lines = []
                for page in pages:
                    facsimile = {"number": page["number"], "url": page["image"]}
                    page_lines = get_page_body_lines(page)
                    facsimile["lines"] = { x['order']: x['region'] for x in page_lines}
                    facsimiles.append(facsimile)
                    lines += page_lines
                lines = get_segment_lines(lines, segments[id], first["id"], last["id"])
                segment_data = get_segment_data(
                    lines, segments[id]
                )  ## todo find what additional information should be includedin each segement dict
                segments[id].update(segment_data)
                ms_units.append(segments[id])

        dto["ms_units"] = ms_units
        dto["facsimiles"] = facsimiles
        data.append(dto)
    return units, data

In [188]:
units, data = pipeline("Mc",  "Mouse and Cat")

In [189]:
CHAPTER = "Mc"
OUT_PATH = f"../apps/web/src/assets/data/collations/{CHAPTER}"

In [190]:
with open(f"{OUT_PATH}/units.json", "w", encoding='utf-8') as write_file:
    json.dump(units, write_file)

In [191]:
def clone_dictionary(original_dict, excluded_fields):
    return {key: value for key, value in original_dict.items() if key not in excluded_fields}

meta = []
for dto in data:
    meta.append(clone_dictionary(dto, ['ms_units']))

In [192]:
with open(f"{OUT_PATH}/meta.json", "w", encoding='utf-8') as write_file:
    json.dump(meta, write_file)

In [193]:
def save_dict_chunks_to_json(data_dict, chunk_size):
    num_chunks = len(next(iter(data_dict.values()))) // chunk_size  # Get length from the first list
    if len(next(iter(data_dict.values()))) % chunk_size != 0:
        num_chunks += 1

    for i in range(num_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size
        chunk_dict = {key: value[start:end] for key, value in data_dict.items()}  # Create chunk dictionary
        filename = f"{start}_to_{end - 1}.json"

        with open(f"{OUT_PATH}/{filename}", 'w', encoding='utf-8') as file:
            json.dump(chunk_dict, file, ensure_ascii=False)

In [194]:
segment_data = {x['siglum']: x['ms_units'] for x in data}
save_dict_chunks_to_json(segment_data, 10)